# Walmart Dynamic Pricing and Revenue Optimization

## Milestone 3.1 — Price Simulation

This notebook uses the trained demand forecasting model to simulate alternative product prices.

For each candidate price, the model predicts expected demand and calculates projected revenue.

The goal is to identify prices that maximize expected revenue while maintaining realistic pricing behavior.

In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

In [2]:
model_path = Path("../models/gradient_boosting_forecast_model.pkl")

forecast_model = joblib.load(model_path)

print("Forecasting model loaded successfully.")

Forecasting model loaded successfully.


In [3]:
data_path = Path("../data/processed/walmart_features.pkl")

pricing_df = pd.read_pickle(data_path)

print(pricing_df.shape)

pricing_df.head()

(4748523, 30)


,date,store_id,item_id,units_sold,lag_1,lag_7,lag_30,rolling_avg_7,rolling_avg_28,sales_change_1,...,is_event,snap_active,temperature_max,temperature_min,precipitation,snowfall,wind_speed_max,cpi,unemployment_rate,federal_funds_rate
30,2011-02-28,CA_1,FOODS_1_001,0,2.0,0.0,3.0,2.000000,1.428571,2.0,...,0,0,16.8,0.9,0.0,0.0,12.1,221.898,9.0,0.16
31,2011-03-01,CA_1,FOODS_1_001,2,0.0,2.0,0.0,2.000000,1.428571,-2.0,...,0,1,16.3,3.0,0.0,0.0,12.5,223.046,9.0,0.14
32,2011-03-02,CA_1,FOODS_1_001,1,2.0,2.0,0.0,2.000000,1.464286,0.0,...,0,1,15.1,4.7,0.4,0.0,7.6,223.046,9.0,0.14
33,2011-03-03,CA_1,FOODS_1_001,7,1.0,2.0,1.0,1.857143,1.357143,-1.0,...,0,1,18.7,9.9,0.0,0.0,11.3,223.046,9.0,0.14
34,2011-03-04,CA_1,FOODS_1_001,1,7.0,4.0,4.0,2.571429,1.535714,3.0,...,0,1,23.1,6.9,0.0,0.0,8.0,223.046,9.0,0.14


## Select Product for Price Simulation

A single product-store observation is selected to demonstrate the dynamic pricing process.

The product's current selling price will serve as the baseline price. Alternative prices will then be simulated to estimate how predicted demand and revenue respond to price changes.

In [4]:
# Select an observation with a valid selling price
pricing_example = (
    pricing_df[
        pricing_df["sell_price"] > 0
    ]
    .sort_values("date")
    .iloc[-1]
    .copy()
)

print("Date:", pricing_example["date"])
print("Store:", pricing_example["store_id"])
print("Product:", pricing_example["item_id"])
print("Current price:", pricing_example["sell_price"])
print("Actual units sold:", pricing_example["units_sold"])

Date: 2016-05-22 00:00:00
Store: CA_1
Product: HOUSEHOLD_2_516
Current price: 5.94
Actual units sold: 0


## Generate Candidate Prices

Alternative prices are generated around the product's current selling price.

These candidate prices will be passed through the forecasting model to estimate how changes in price affect predicted demand and revenue.

In [5]:
# Current selling price
current_price = pricing_example["sell_price"]

# Test prices from 20% below to 20% above the current price
price_multipliers = np.arange(0.80, 1.21, 0.05)

candidate_prices = np.round(
    current_price * price_multipliers,
    2
)

candidate_prices

array([4.75, 5.05, 5.35, 5.64, 5.94, 6.24, 6.53, 6.83, 7.13])

## Simulate Demand at Alternative Prices

Each candidate price is substituted into the product's feature set and passed through the trained Gradient Boosting forecasting model.

The resulting predictions estimate demand at each potential selling price.

In [6]:
# Same features used to train the forecasting model
feature_columns = [
    "lag_1",
    "lag_7",
    "lag_30",
    "rolling_avg_7",
    "rolling_avg_28",
    "sales_change_1",
    "sales_change_7_28",
    "sell_price",
    "price_lag_1",
    "price_change_1",
    "price_pct_change_1",
    "day_of_week",
    "month",
    "year",
    "day_of_month",
    "is_weekend",
    "is_event",
    "snap_active",
    "temperature_max",
    "temperature_min",
    "precipitation",
    "snowfall",
    "wind_speed_max",
    "cpi",
    "unemployment_rate",
    "federal_funds_rate"
]

print("Number of features:", len(feature_columns))

Number of features: 26


In [7]:
import joblib
from pathlib import Path

model_path = Path("../models/gradient_boosting_forecast_model.pkl")

hgb_model = joblib.load(model_path)

print("Forecasting model loaded successfully.")

simulation_rows = []

for candidate_price in candidate_prices:

    # Copy the original product observation
    simulated_row = pricing_example.copy()

    # Replace current price with candidate price
    simulated_row["sell_price"] = candidate_price

    # Select the same features used to train the model
    X_simulated = pd.DataFrame(
        [simulated_row[feature_columns].values],
        columns=feature_columns
    )

    # Predict demand
    predicted_demand = hgb_model.predict(X_simulated)[0]

    # Demand cannot be negative
    predicted_demand = max(0, predicted_demand)

    simulation_rows.append({
        "price": candidate_price,
        "predicted_demand": predicted_demand
    })

price_simulation = pd.DataFrame(simulation_rows)

price_simulation

Forecasting model loaded successfully.


,price,predicted_demand
0,4.75,0.247944
1,5.05,0.247944
2,5.35,0.245164
3,5.64,0.245164
4,5.94,0.245164
5,6.24,0.245164
6,6.53,0.245164
7,6.83,0.245164
8,7.13,0.245164


## Revenue Simulation

Projected revenue is calculated for each candidate price using:

**Projected Revenue = Candidate Price × Predicted Demand**

The price with the highest projected revenue is selected as the recommended price.

In [8]:
price_simulation["predicted_revenue"] = (
    price_simulation["price"]
    * price_simulation["predicted_demand"]
)

price_simulation

,price,predicted_demand,predicted_revenue
0,4.75,0.247944,1.177734
1,5.05,0.247944,1.252117
2,5.35,0.245164,1.311629
3,5.64,0.245164,1.382727
4,5.94,0.245164,1.456276
5,6.24,0.245164,1.529826
6,6.53,0.245164,1.600923
7,6.83,0.245164,1.674473
8,7.13,0.245164,1.748022


In [9]:
optimal_row = price_simulation.loc[
    price_simulation["predicted_revenue"].idxmax()
]

print(f"Current price: ${current_price:.2f}")
print(f"Recommended price: ${optimal_row['price']:.2f}")
print(
    f"Predicted demand at recommended price: "
    f"{optimal_row['predicted_demand']:.4f}"
)
print(
    f"Projected revenue at recommended price: "
    f"${optimal_row['predicted_revenue']:.4f}"
)

Current price: $5.94
Recommended price: $7.13
Predicted demand at recommended price: 0.2452
Projected revenue at recommended price: $1.7480


## Evaluate Pricing Recommendation

The recommended price is compared with the current selling price to estimate the potential change in revenue.

This provides a business-oriented measure of the pricing recommendation.

In [10]:
# Find the simulation closest to the current price
current_row = price_simulation.iloc[
    (price_simulation["price"] - current_price).abs().argsort()[:1]
].iloc[0]

current_revenue = current_row["predicted_revenue"]
recommended_revenue = optimal_row["predicted_revenue"]

revenue_change = recommended_revenue - current_revenue

revenue_change_pct = (
    revenue_change / current_revenue
) * 100

print(f"Current price: ${current_price:.2f}")
print(f"Recommended price: ${optimal_row['price']:.2f}")

print()

print(f"Projected revenue at current price: ${current_revenue:.4f}")
print(f"Projected revenue at recommended price: ${recommended_revenue:.4f}")

print()

print(f"Projected revenue increase: ${revenue_change:.4f}")
print(f"Projected revenue improvement: {revenue_change_pct:.2f}%")

Current price: $5.94
Recommended price: $7.13

Projected revenue at current price: $1.4563
Projected revenue at recommended price: $1.7480

Projected revenue increase: $0.2917
Projected revenue improvement: 20.03%


## Pricing Guardrails

To prevent unrealistic price recommendations, candidate prices are constrained
to a maximum 10% increase or decrease from the current selling price.

This creates a more conservative pricing strategy suitable for real-world
retail decision support.

In [11]:
# Define pricing guardrails
max_price_increase = 0.10
max_price_decrease = 0.10

minimum_allowed_price = current_price * (1 - max_price_decrease)
maximum_allowed_price = current_price * (1 + max_price_increase)

# Keep only prices within the allowed range
guardrail_simulation = price_simulation[
    (price_simulation["price"] >= minimum_allowed_price)
    & (price_simulation["price"] <= maximum_allowed_price)
].copy()

# Select best price within guardrails
guardrail_optimal = guardrail_simulation.loc[
    guardrail_simulation["predicted_revenue"].idxmax()
]

guardrail_simulation

,price,predicted_demand,predicted_revenue
2,5.35,0.245164,1.311629
3,5.64,0.245164,1.382727
4,5.94,0.245164,1.456276
5,6.24,0.245164,1.529826
6,6.53,0.245164,1.600923


## Final Pricing Recommendation

The optimal price within the defined pricing guardrails is selected and compared
with the current selling price.

This produces the final pricing recommendation and estimated revenue improvement.

In [12]:
final_price = guardrail_optimal["price"]
final_demand = guardrail_optimal["predicted_demand"]
final_revenue = guardrail_optimal["predicted_revenue"]

final_revenue_change = final_revenue - current_revenue

final_revenue_change_pct = (
    final_revenue_change / current_revenue
) * 100

price_change_pct = (
    (final_price - current_price) / current_price
) * 100

print(f"Current price: ${current_price:.2f}")
print(f"Recommended price: ${final_price:.2f}")
print(f"Price change: {price_change_pct:.2f}%")

print()

print(f"Predicted demand: {final_demand:.4f}")
print(f"Current projected revenue: ${current_revenue:.4f}")
print(f"Recommended projected revenue: ${final_revenue:.4f}")

print()

print(f"Projected revenue improvement: {final_revenue_change_pct:.2f}%")

Current price: $5.94
Recommended price: $6.53
Price change: 9.93%

Predicted demand: 0.2452
Current projected revenue: $1.4563
Recommended projected revenue: $1.6009

Projected revenue improvement: 9.93%


## Reusable Pricing Recommendation Function

The pricing logic is converted into a reusable function so that different product-store observations can be evaluated automatically.

The function generates candidate prices within the allowed pricing guardrails, predicts demand at each price, calculates projected revenue, and returns the best recommendation.

In [13]:
def recommend_price(
    observation,
    model,
    feature_columns,
    max_increase=0.10,
    max_decrease=0.10,
    step=0.05
):
    # Current selling price
    current_price = observation["sell_price"]

    # Define exact guardrail limits
    min_price = current_price * (1 - max_decrease)
    max_price = current_price * (1 + max_increase)

    # Generate candidate price multipliers
    multipliers = np.arange(
        1 - max_decrease,
        1 + max_increase + step,
        step
    )

    # Generate candidate prices
    candidate_prices = np.round(
        current_price * multipliers,
        2
    )

    # Enforce exact pricing guardrails
    candidate_prices = candidate_prices[
        (candidate_prices >= min_price)
        & (candidate_prices <= max_price)
    ]

    simulation_rows = []

    # Simulate demand and revenue at each candidate price
    for candidate_price in candidate_prices:

        simulated_row = observation.copy()

        # Replace only the current selling price
        simulated_row["sell_price"] = candidate_price

        # Select the exact same features used during model training
        X_simulated = pd.DataFrame(
            [simulated_row[feature_columns].values],
            columns=feature_columns
        )

        # Predict demand
        predicted_demand = model.predict(
            X_simulated
        )[0]

        # Prevent negative demand predictions
        predicted_demand = max(
            0,
            predicted_demand
        )

        # Calculate projected revenue
        predicted_revenue = (
            candidate_price
            * predicted_demand
        )

        simulation_rows.append({
            "price": candidate_price,
            "predicted_demand": predicted_demand,
            "predicted_revenue": predicted_revenue
        })

    # Convert simulation results to DataFrame
    simulation = pd.DataFrame(
        simulation_rows
    )

    # Select candidate with highest projected revenue
    optimal = simulation.loc[
        simulation["predicted_revenue"].idxmax()
    ]

    return simulation, optimal

In [14]:
simulation_test, recommendation_test = recommend_price(
    pricing_example,
    hgb_model,
    feature_columns
)

print("PRICE SIMULATION")
display(simulation_test)

print("\nRECOMMENDATION")
display(recommendation_test)

PRICE SIMULATION


,price,predicted_demand,predicted_revenue
0,5.35,0.245164,1.311629
1,5.64,0.245164,1.382727
2,5.94,0.245164,1.456276
3,6.24,0.245164,1.529826
4,6.53,0.245164,1.600923



RECOMMENDATION


price                6.530000
predicted_demand     0.245164
predicted_revenue    1.600923
Name: 4, dtype: float64

## Multi-Product Pricing Recommendations

The reusable pricing function is applied across multiple product observations.

For each product, the engine compares candidate prices within the allowed guardrails and returns the price that maximizes projected revenue.

In [15]:
# Select the most recent observation for a sample of products
latest_date = pricing_df["date"].max()

product_sample = (
    pricing_df[
        (pricing_df["date"] == latest_date)
        & (pricing_df["sell_price"] > 0)
    ]
    .drop_duplicates(
        subset=["store_id", "item_id"]
    )
    .head(20)
    .copy()
)

print("Products selected:", len(product_sample))

product_sample[
    [
        "date",
        "store_id",
        "item_id",
        "sell_price",
        "units_sold"
    ]
]

Products selected: 20


,date,store_id,item_id,sell_price,units_sold
1940,2016-05-22,CA_1,FOODS_1_001,2.24,0
3881,2016-05-22,CA_1,FOODS_1_002,9.48,2
5822,2016-05-22,CA_1,FOODS_1_003,3.23,1
7763,2016-05-22,CA_1,FOODS_1_004,1.96,4
9704,2016-05-22,CA_1,FOODS_1_005,3.54,1
11645,2016-05-22,CA_1,FOODS_1_006,2.48,0
13586,2016-05-22,CA_1,FOODS_1_008,3.54,0
15527,2016-05-22,CA_1,FOODS_1_009,2.24,3
17468,2016-05-22,CA_1,FOODS_1_010,5.64,0
19409,2016-05-22,CA_1,FOODS_1_011,2.68,1


## Generate Multi-Product Price Recommendations

The pricing engine is applied to each selected product-store observation.

For every product, the system compares alternative prices within the pricing
guardrails and selects the price with the highest projected revenue.

In [16]:
recommendations = []

for _, observation in product_sample.iterrows():

    # Run pricing engine
    simulation, optimal = recommend_price(
        observation,
        hgb_model,
        feature_columns
    )

    current_price = observation["sell_price"]

    # Find candidate closest to current price
    current_result = simulation.iloc[
        (simulation["price"] - current_price)
        .abs()
        .argsort()[:1]
    ].iloc[0]

    current_revenue = current_result["predicted_revenue"]
    recommended_revenue = optimal["predicted_revenue"]

    # Calculate price change
    price_change_pct = (
        (optimal["price"] - current_price)
        / current_price
    ) * 100

    # Calculate projected revenue improvement
    if current_revenue > 0:
        revenue_improvement_pct = (
            (recommended_revenue - current_revenue)
            / current_revenue
        ) * 100
    else:
        revenue_improvement_pct = 0

    recommendations.append({
        "store_id": observation["store_id"],
        "item_id": observation["item_id"],
        "current_price": current_price,
        "recommended_price": optimal["price"],
        "price_change_pct": price_change_pct,
        "predicted_demand": optimal["predicted_demand"],
        "current_revenue": current_revenue,
        "recommended_revenue": recommended_revenue,
        "revenue_improvement_pct": revenue_improvement_pct
    })

pricing_recommendations = pd.DataFrame(recommendations)

pricing_recommendations

,store_id,item_id,current_price,recommended_price,price_change_pct,predicted_demand,current_revenue,recommended_revenue,revenue_improvement_pct
0,CA_1,FOODS_1_001,2.24,2.46,9.821429,0.852366,1.909300,2.096821,9.821429
1,CA_1,FOODS_1_002,9.48,9.95,4.957806,0.855799,8.112971,8.515197,4.957806
2,CA_1,FOODS_1_003,3.23,3.55,9.907121,0.821968,2.690007,2.917987,8.475068
3,CA_1,FOODS_1_004,1.96,2.06,5.102041,3.670960,7.206472,7.562177,4.935911
4,CA_1,FOODS_1_005,3.54,3.89,9.887006,1.660561,5.918416,6.459583,9.143774
5,CA_1,FOODS_1_006,2.48,2.60,4.838710,3.014135,7.475056,7.836752,4.838710
6,CA_1,FOODS_1_008,3.54,3.89,9.887006,0.324507,1.188783,1.262330,6.186790
7,CA_1,FOODS_1_009,2.24,2.46,9.821429,1.583994,3.548146,3.896625,9.821429
8,CA_1,FOODS_1_010,5.64,6.20,9.929078,0.406539,2.292882,2.520544,9.929078
9,CA_1,FOODS_1_011,2.68,2.81,4.850746,0.840117,2.251514,2.360729,4.850746


## Pricing Recommendation Summary

The pricing recommendations are summarized to evaluate the overall impact of
the optimization strategy across the selected products.

In [17]:
summary = pd.Series({
    "Products analyzed": len(pricing_recommendations),

    "Average current price":
        pricing_recommendations["current_price"].mean(),

    "Average recommended price":
        pricing_recommendations["recommended_price"].mean(),

    "Average price change (%)":
        pricing_recommendations["price_change_pct"].mean(),

    "Average projected revenue improvement (%)":
        pricing_recommendations["revenue_improvement_pct"].mean(),

    "Products with price increase":
        (pricing_recommendations["recommended_price"]
         > pricing_recommendations["current_price"]).sum(),

    "Products with price decrease":
        (pricing_recommendations["recommended_price"]
         < pricing_recommendations["current_price"]).sum(),

    "Products with no price change":
        (pricing_recommendations["recommended_price"]
         == pricing_recommendations["current_price"]).sum()
})

summary

Products analyzed                            20.000000
Average current price                         2.937500
Average recommended price                     3.156500
Average price change (%)                      7.188205
Average projected revenue improvement (%)     7.079686
Products with price increase                 20.000000
Products with price decrease                  0.000000
Products with no price change                 0.000000
dtype: float64

## Save Pricing Recommendations

The generated pricing recommendations are saved for downstream use by the
FastAPI service and Streamlit decision-support dashboard.

In [18]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

pricing_path = output_dir / "pricing_recommendations.pkl"

pricing_recommendations.to_pickle(pricing_path)

print(f"Pricing recommendations saved to: {pricing_path}")
print(f"Rows: {len(pricing_recommendations):,}")
print(f"Columns: {len(pricing_recommendations.columns)}")

Pricing recommendations saved to: ..\data\processed\pricing_recommendations.pkl
Rows: 20
Columns: 9


## Price Sensitivity Analysis

Before finalizing the pricing engine, the relationship between historical
selling prices and product demand is analyzed.

This helps determine whether changes in price are associated with meaningful
changes in units sold and whether the forecasting model captures sufficient
price sensitivity for dynamic pricing.

In [19]:
price_sensitivity = (
    pricing_df[
        ["sell_price", "units_sold"]
    ]
    .corr()
)

price_sensitivity

,sell_price,units_sold
sell_price,1.000000,-0.160075
units_sold,-0.160075,1.000000


In [20]:
price_demand_correlation = price_sensitivity.loc[
    "sell_price",
    "units_sold"
]

print(
    f"Price-demand correlation: "
    f"{price_demand_correlation:.4f}"
)

Price-demand correlation: -0.1601


## Product-Level Price Sensitivity

Overall price-demand correlation may hide differences between individual
products. Price sensitivity is therefore calculated separately for each
store-product combination.

This provides a more realistic view of how demand responds to historical
price changes.

In [21]:
product_price_sensitivity = (
    pricing_df
    .groupby(["store_id", "item_id"])
    .apply(
        lambda x: x["sell_price"].corr(x["units_sold"]),
        include_groups=False
    )
    .reset_index(name="price_demand_correlation")
)

product_price_sensitivity.head(20)

c:\Users\7800XT\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\7800XT\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,store_id,item_id,price_demand_correlation
0,CA_1,FOODS_1_001,-9.685786e-02
1,CA_1,FOODS_1_002,-1.269571e-01
2,CA_1,FOODS_1_003,-5.759220e-02
3,CA_1,FOODS_1_004,-1.472962e-01
4,CA_1,FOODS_1_005,2.113913e-02
5,CA_1,FOODS_1_006,-1.033616e-01
6,CA_1,FOODS_1_008,2.019164e-01
7,CA_1,FOODS_1_009,-3.411931e-02
8,CA_1,FOODS_1_010,1.748949e-02
9,CA_1,FOODS_1_011,1.523189e-01


In [22]:
print(
    product_price_sensitivity[
        "price_demand_correlation"
    ].describe()
)

print()

negative_pct = (
    product_price_sensitivity[
        "price_demand_correlation"
    ].lt(0).mean() * 100
)

positive_pct = (
    product_price_sensitivity[
        "price_demand_correlation"
    ].gt(0).mean() * 100
)

print(f"Products with negative correlation: {negative_pct:.2f}%")
print(f"Products with positive correlation: {positive_pct:.2f}%")

count    2787.000000
mean       -0.044708
std         0.124929
min        -0.727074
25%        -0.103007
50%        -0.006387
75%         0.001523
max         0.775334
Name: price_demand_correlation, dtype: float64

Products with negative correlation: 58.22%
Products with positive correlation: 32.96%


## Estimate Price Elasticity

Price elasticity measures how strongly product demand responds to changes
in selling price.

Unlike simple correlation, elasticity measures the percentage change in
demand associated with a percentage change in price. This provides a more
useful signal for evaluating pricing decisions.

In [23]:
# Keep observations with positive price and demand
elasticity_df = pricing_df[
    (pricing_df["sell_price"] > 0) &
    (pricing_df["units_sold"] > 0)
].copy()

# Log transformations
elasticity_df["log_price"] = np.log(
    elasticity_df["sell_price"]
)

elasticity_df["log_demand"] = np.log(
    elasticity_df["units_sold"]
)

print("Rows available:", f"{len(elasticity_df):,}")

elasticity_df[
    [
        "store_id",
        "item_id",
        "sell_price",
        "units_sold",
        "log_price",
        "log_demand"
    ]
].head()

Rows available: 2,122,462


,store_id,item_id,sell_price,units_sold,log_price,log_demand
31,CA_1,FOODS_1_001,2.0,2,0.693147,0.693147
32,CA_1,FOODS_1_001,2.0,1,0.693147,0.000000
33,CA_1,FOODS_1_001,2.0,7,0.693147,1.945910
34,CA_1,FOODS_1_001,2.0,1,0.693147,0.000000
35,CA_1,FOODS_1_001,2.0,2,0.693147,0.693147


## Product-Level Price Elasticity

Price elasticity is estimated for each store-product combination using
a log-log regression of demand on selling price.

The resulting elasticity coefficient estimates the percentage change in
demand associated with a 1% change in price.

In [24]:
from sklearn.linear_model import LinearRegression

elasticity_results = []

for (store_id, item_id), group in elasticity_df.groupby(
    ["store_id", "item_id"]
):

    # Need enough observations and more than one unique price
    if len(group) >= 30 and group["sell_price"].nunique() >= 2:

        X = group[["log_price"]]
        y = group["log_demand"]

        elasticity_model = LinearRegression()

        elasticity_model.fit(
            X,
            y
        )

        elasticity = elasticity_model.coef_[0]

        elasticity_results.append({
            "store_id": store_id,
            "item_id": item_id,
            "price_elasticity": elasticity,
            "observations": len(group),
            "unique_prices": group["sell_price"].nunique()
        })

product_elasticity = pd.DataFrame(elasticity_results)

print(
    "Products with estimated elasticity:",
    f"{len(product_elasticity):,}"
)

product_elasticity.head(20)

Products with estimated elasticity: 2,163


,store_id,item_id,price_elasticity,observations,unique_prices
0,CA_1,FOODS_1_001,-1.035095,826,2
1,CA_1,FOODS_1_002,-0.986380,641,3
2,CA_1,FOODS_1_003,-0.504927,895,2
3,CA_1,FOODS_1_004,-1.020204,1323,3
4,CA_1,FOODS_1_005,-1.492090,877,3
5,CA_1,FOODS_1_006,-1.063240,1153,3
6,CA_1,FOODS_1_008,0.824583,287,2
7,CA_1,FOODS_1_009,-0.937006,503,4
8,CA_1,FOODS_1_010,1.009332,231,10
9,CA_1,FOODS_1_011,0.446756,567,4


## Evaluate Price Elasticity Estimates

The distribution of estimated price elasticities is evaluated before the
values are incorporated into the pricing engine.

Extreme or economically implausible elasticity estimates may result from
limited historical price variation, promotions, or other confounding factors.

In [25]:
print(
    product_elasticity["price_elasticity"].describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print()

negative_elasticity_pct = (
    product_elasticity["price_elasticity"].lt(0).mean()
    * 100
)

positive_elasticity_pct = (
    product_elasticity["price_elasticity"].gt(0).mean()
    * 100
)

print(
    f"Negative elasticity: "
    f"{negative_elasticity_pct:.2f}%"
)

print(
    f"Positive elasticity: "
    f"{positive_elasticity_pct:.2f}%"
)

count    2163.000000
mean       -1.163567
std        12.263198
min      -403.489838
1%        -34.420117
5%         -7.307348
10%        -4.050479
25%        -1.868839
50%        -0.556516
75%         0.429548
90%         2.291581
95%         4.994916
99%        18.443229
max       116.918705
Name: price_elasticity, dtype: float64

Negative elasticity: 65.88%
Positive elasticity: 34.12%


## Clean Price Elasticity Estimates

Extreme and positive elasticity estimates may be caused by limited price
variation, promotions, seasonality, or noisy historical relationships.

To prevent unstable pricing recommendations, only economically plausible
negative elasticity estimates between -3.0 and -0.1 are retained.

In [26]:
valid_elasticity = product_elasticity[
    product_elasticity["price_elasticity"].between(
        -3.0,
        -0.1
    )
].copy()

print(
    "Original elasticity estimates:",
    f"{len(product_elasticity):,}"
)

print(
    "Valid elasticity estimates:",
    f"{len(valid_elasticity):,}"
)

print(
    "Percent retained:",
    f"{len(valid_elasticity) / len(product_elasticity) * 100:.2f}%"
)

print()

print(
    valid_elasticity["price_elasticity"].describe()
)

Original elasticity estimates: 2,163
Valid elasticity estimates: 1,048
Percent retained: 48.45%

count    1048.000000
mean       -1.188735
std         0.780305
min        -2.998923
25%        -1.725190
50%        -0.999793
75%        -0.526302
max        -0.100038
Name: price_elasticity, dtype: float64


## Attach Price Elasticity to Products

Validated price elasticity estimates are merged with product observations
so the pricing engine can account for how individual products historically
respond to price changes.

In [27]:
pricing_with_elasticity = pricing_df.merge(
    valid_elasticity[
        [
            "store_id",
            "item_id",
            "price_elasticity"
        ]
    ],
    on=["store_id", "item_id"],
    how="left"
)

print("Rows:", f"{len(pricing_with_elasticity):,}")

print(
    "Rows with valid elasticity:",
    f"{pricing_with_elasticity['price_elasticity'].notna().sum():,}"
)

print(
    "Unique products with valid elasticity:",
    pricing_with_elasticity[
        "price_elasticity"
    ].notna().groupby(
        [
            pricing_with_elasticity["store_id"],
            pricing_with_elasticity["item_id"]
        ]
    ).any().sum()
)

Rows: 4,748,523
Rows with valid elasticity: 1,782,846
Unique products with valid elasticity: 1048


## Elasticity-Adjusted Pricing Engine

The pricing engine is enhanced by combining the machine-learning demand forecast
with product-level price elasticity.

For each candidate price:

1. The forecasting model estimates baseline demand.
2. Product elasticity adjusts that demand based on the percentage price change.
3. Projected revenue is calculated.
4. The price with the highest projected revenue is selected.

This creates more realistic pricing behavior than relying on the forecasting
model's price feature alone.

In [28]:
def recommend_price_with_elasticity(
    observation,
    model,
    feature_columns,
    max_increase=0.10,
    max_decrease=0.10,
    step=0.05,
    default_elasticity=-1.0
):
    current_price = observation["sell_price"]

    min_price = current_price * (1 - max_decrease)
    max_price = current_price * (1 + max_increase)

    # Use product elasticity if available
    elasticity = observation.get(
        "price_elasticity",
        np.nan
    )

    # Fall back to conservative default
    if pd.isna(elasticity):
        elasticity = default_elasticity

    multipliers = np.arange(
        1 - max_decrease,
        1 + max_increase + step,
        step
    )

    candidate_prices = np.round(
        current_price * multipliers,
        2
    )

    # Enforce exact pricing guardrails
    candidate_prices = candidate_prices[
        (candidate_prices >= min_price)
        & (candidate_prices <= max_price)
    ]

    simulation_rows = []

    for candidate_price in candidate_prices:

        simulated_row = observation.copy()

        simulated_row["sell_price"] = candidate_price

        X_simulated = pd.DataFrame(
            [simulated_row[feature_columns].values],
            columns=feature_columns
        )

        # ML baseline demand prediction
        baseline_demand = model.predict(
            X_simulated
        )[0]

        baseline_demand = max(
            0,
            baseline_demand
        )

        # Calculate relative price change
        price_ratio = (
            candidate_price / current_price
        )

        # Elasticity demand adjustment
        adjusted_demand = (
            baseline_demand
            * (price_ratio ** elasticity)
        )

        adjusted_demand = max(
            0,
            adjusted_demand
        )

        predicted_revenue = (
            candidate_price
            * adjusted_demand
        )

        simulation_rows.append({
            "price": candidate_price,
            "elasticity": elasticity,
            "baseline_demand": baseline_demand,
            "adjusted_demand": adjusted_demand,
            "predicted_revenue": predicted_revenue
        })

    simulation = pd.DataFrame(
        simulation_rows
    )

    optimal = simulation.loc[
        simulation["predicted_revenue"].idxmax()
    ]

    return simulation, optimal

In [29]:
elasticity_example = (
    pricing_with_elasticity
    .dropna(subset=["price_elasticity"])
    .sort_values("date")
    .iloc[-1]
    .copy()
)

print("Product:", elasticity_example["item_id"])
print("Current price:", elasticity_example["sell_price"])
print("Elasticity:", elasticity_example["price_elasticity"])

Product: HOUSEHOLD_2_505
Current price: 4.97
Elasticity: -0.7322073208656074


In [30]:
elasticity_simulation, elasticity_recommendation = (
    recommend_price_with_elasticity(
        elasticity_example,
        hgb_model,
        feature_columns
    )
)

display(elasticity_simulation)

print("\nRECOMMENDATION")
display(elasticity_recommendation)

,price,elasticity,baseline_demand,adjusted_demand,predicted_revenue
0,4.72,-0.732207,0.377403,0.391938,1.849946
1,4.97,-0.732207,0.377403,0.377403,1.875692
2,5.22,-0.732207,0.377403,0.364082,1.900506



RECOMMENDATION


price                5.220000
elasticity          -0.732207
baseline_demand      0.377403
adjusted_demand      0.364082
predicted_revenue    1.900506
Name: 2, dtype: float64

## Multi-Product Elasticity-Aware Pricing Recommendations

The elasticity-adjusted pricing engine is applied across multiple products with valid elasticity estimates.

For each product, the engine evaluates candidate prices within the pricing guardrails, adjusts predicted demand using product-level elasticity, calculates projected revenue, and selects the price that maximizes expected revenue.

In [31]:
# Use the most recent observation for each product with valid elasticity
latest_date = pricing_with_elasticity["date"].max()

elasticity_product_sample = (
    pricing_with_elasticity[
        (pricing_with_elasticity["date"] == latest_date)
        & (pricing_with_elasticity["price_elasticity"].notna())
        & (pricing_with_elasticity["sell_price"] > 0)
    ]
    .drop_duplicates(
        subset=["store_id", "item_id"]
    )
    .copy()
)

print(
    "Products available:",
    f"{len(elasticity_product_sample):,}"
)

elasticity_product_sample[
    [
        "store_id",
        "item_id",
        "sell_price",
        "price_elasticity",
        "units_sold"
    ]
].head(20)

Products available: 1,048


,store_id,item_id,sell_price,price_elasticity,units_sold
1910,CA_1,FOODS_1_001,2.24,-1.035095,0
3821,CA_1,FOODS_1_002,9.48,-0.986380,2
5732,CA_1,FOODS_1_003,3.23,-0.504927,1
7274,CA_1,FOODS_1_004,1.96,-1.020204,4
9185,CA_1,FOODS_1_005,3.54,-1.492090,1
11096,CA_1,FOODS_1_006,2.48,-1.063240,0
14320,CA_1,FOODS_1_009,2.24,-0.937006,3
19448,CA_1,FOODS_1_012,5.64,-2.632951,8
21359,CA_1,FOODS_1_013,1.50,-0.534720,0
24371,CA_1,FOODS_1_015,3.48,-1.929410,0


## Generate Elasticity-Aware Pricing Recommendations

The pricing optimization function is applied to all products with valid elasticity estimates.

For each product, candidate prices are evaluated within the pricing guardrails. Demand is adjusted according to the product's estimated price elasticity, and the price producing the highest projected revenue is selected.

In [32]:
elasticity_recommendations = []

for _, product_row in elasticity_product_sample.iterrows():

    # Run elasticity-aware pricing optimization
    simulation, recommendation = recommend_price_with_elasticity(
        product_row,
        hgb_model,
        feature_columns
    )

    current_price = product_row["sell_price"]
    recommended_price = recommendation["price"]

    # Current predicted demand from the forecasting model
    X_current = pd.DataFrame(
        [product_row[feature_columns].values],
        columns=feature_columns
    )

    current_demand = hgb_model.predict(X_current)[0]
    current_demand = max(0, current_demand)

    current_revenue = current_price * current_demand
    recommended_revenue = recommendation["predicted_revenue"]

    # Calculate projected revenue improvement
    if current_revenue > 0:
        revenue_improvement_pct = (
            (recommended_revenue - current_revenue)
            / current_revenue
        ) * 100
    else:
        revenue_improvement_pct = 0

    elasticity_recommendations.append({
        "store_id": product_row["store_id"],
        "item_id": product_row["item_id"],
        "current_price": current_price,
        "recommended_price": recommended_price,
        "price_change_pct": (
            (recommended_price - current_price)
            / current_price
        ) * 100,
        "price_elasticity": product_row["price_elasticity"],
        "current_predicted_demand": current_demand,
        "adjusted_demand": recommendation["adjusted_demand"],
        "current_revenue": current_revenue,
        "recommended_revenue": recommended_revenue,
        "revenue_improvement_pct": revenue_improvement_pct
    })

elasticity_recommendations = pd.DataFrame(
    elasticity_recommendations
)

print(
    "Recommendations generated:",
    f"{len(elasticity_recommendations):,}"
)

elasticity_recommendations.head(20)

Recommendations generated: 1,048


,store_id,item_id,current_price,recommended_price,price_change_pct,price_elasticity,current_predicted_demand,adjusted_demand,current_revenue,recommended_revenue,revenue_improvement_pct
0,CA_1,FOODS_1_001,2.24,2.02,-9.821429,-1.035095,0.852366,0.948634,1.909300,1.916240,0.363466
1,CA_1,FOODS_1_002,9.48,9.95,4.957806,-0.986380,0.855799,0.815912,8.112971,8.118320,0.065925
2,CA_1,FOODS_1_003,3.23,3.55,9.907121,-0.504927,0.832820,0.783682,2.690007,2.782072,3.422467
3,CA_1,FOODS_1_004,1.96,1.86,-5.102041,-1.020204,3.676771,3.887357,7.206472,7.230484,0.333200
4,CA_1,FOODS_1_005,3.54,3.19,-9.887006,-1.492090,1.671869,1.965501,5.918416,6.269947,5.939608
5,CA_1,FOODS_1_006,2.48,2.36,-4.838710,-1.063240,3.014135,3.177347,7.475056,7.498538,0.314145
6,CA_1,FOODS_1_009,2.24,2.46,9.821429,-0.937006,1.583994,1.450873,3.548146,3.569148,0.591907
7,CA_1,FOODS_1_012,5.64,5.08,-9.929078,-2.632951,3.758704,4.953768,21.199089,25.165141,18.708597
8,CA_1,FOODS_1_013,1.50,1.65,10.000000,-0.534720,1.104311,1.049440,1.656466,1.731577,4.534390
9,CA_1,FOODS_1_015,3.48,3.31,-4.885057,-1.929410,4.758636,5.241427,16.560053,17.349125,4.764909


## Pricing Recommendation Summary

The elasticity-aware pricing recommendations are summarized to evaluate the overall pricing strategy, including the number of price increases and decreases, average price changes, and projected revenue improvement.

In [33]:
summary = pd.Series({
    "Products analyzed":
        len(elasticity_recommendations),

    "Average current price":
        elasticity_recommendations["current_price"].mean(),

    "Average recommended price":
        elasticity_recommendations["recommended_price"].mean(),

    "Average price change (%)":
        elasticity_recommendations["price_change_pct"].mean(),

    "Average elasticity":
        elasticity_recommendations["price_elasticity"].mean(),

    "Average revenue improvement (%)":
        elasticity_recommendations["revenue_improvement_pct"].mean(),

    "Products with price increase":
        (
            elasticity_recommendations["recommended_price"]
            > elasticity_recommendations["current_price"]
        ).sum(),

    "Products with price decrease":
        (
            elasticity_recommendations["recommended_price"]
            < elasticity_recommendations["current_price"]
        ).sum(),

    "Products with no price change":
        (
            elasticity_recommendations["recommended_price"]
            == elasticity_recommendations["current_price"]
        ).sum()
})

summary

Products analyzed                  1048.000000
Average current price                 4.439132
Average recommended price             4.449017
Average price change (%)             -0.208507
Average elasticity                   -1.188735
Average revenue improvement (%)       4.434435
Products with price increase        492.000000
Products with price decrease        542.000000
Products with no price change        14.000000
dtype: float64

## Save Final Elasticity-Aware Pricing Recommendations

The final product-level pricing recommendations are saved for downstream use by the FastAPI service and Streamlit decision-support dashboard.

In [34]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

pricing_output_path = (
    output_dir / "elasticity_pricing_recommendations.pkl"
)

elasticity_recommendations.to_pickle(
    pricing_output_path
)

print(
    f"Pricing recommendations saved to: "
    f"{pricing_output_path}"
)

print(
    "Rows:",
    f"{len(elasticity_recommendations):,}"
)

print(
    "Columns:",
    len(elasticity_recommendations.columns)
)

Pricing recommendations saved to: ..\data\processed\elasticity_pricing_recommendations.pkl
Rows: 1,048
Columns: 11
